In [1]:
# Resolve paths consistently when launched from the root or notebooks/.
from pathlib import Path
import os
import sys

PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "topic_embeddings").is_dir() and (p / "notebooks").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Launch this notebook from within the derStandard project.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)


# Compare topic embeddings with mean comment embeddings

How closely does SGNS rank topic pairs by similarity compared with their mean comment vectors? Match the full three-level topic path, then compute a cosine similarity matrix in each embedding space for the shared topics.

In [2]:
import pandas as pd
from topic_embeddings.evaluate_embedding_similarity import evaluate_embedding_similarity

TOPIC_PATH = Path("results/sgns/topic_embeddings_normalized.csv")
MEANS_PATH = Path("data/processed/comment_embedding_means.csv")
topics = pd.read_csv(TOPIC_PATH, keep_default_na=False)
means = pd.read_csv(MEANS_PATH, keep_default_na=False)

topic_cosines, mean_cosines, rho = evaluate_embedding_similarity(topics, means)
n_shared = len(topic_cosines)
print(f"Shared topics: {n_shared:,}")
print(f"SGNS topics without comment means: {len(topics) - n_shared:,}")
print(f"Comment-mean topics without SGNS vectors: {len(means) - n_shared:,}")

Shared topics: 1,000
SGNS topics without comment means: 0
Comment-mean topics without SGNS vectors: 1,077


## Inspect one topic pair

Both matrices use the same topic paths for their rows and columns. Change the row and column positions to inspect another pair.

In [3]:
i, j = 0, 1
for pos in (i, j):
    print(" / ".join(level for level in topic_cosines.index[pos] if level))
print(f"SGNS cosine:          {topic_cosines.iat[i, j]:.6f}")
print(f"Comment-means cosine: {mean_cosines.iat[i, j]:.6f}")

diskurs / debatten / kommentare-der-anderen
diskurs / kolumnen / rauscher
SGNS cosine:          0.741803
Comment-means cosine: 0.997398


## Spearman rank correlation

Each distinct unordered pair contributes once, from the upper triangle above the diagonal. Positive values indicate that pairs with higher SGNS similarity tend to have higher comment-mean similarity.

In [4]:
print(f"Unique topic pairs: {n_shared * (n_shared - 1) // 2:,}")
print(f"Spearman rho: {rho:.6f}")

Unique topic pairs: 499,500
Spearman rho: 0.336557
